In [1]:
%pip install chromadb sentence-transformers pandas

You should consider upgrading via the '/Users/seoyeonkim/customs-rag/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import chromadb

# (참고) 원본 노트북에는 client를 정의하는 셀이 보이지 않아 추가했습니다.
# 이미 다른 곳에서 정의해서 쓰고 있다면 이 셀은 건너뛰고 기존 client를 그대로 쓰면 됩니다.
client = chromadb.PersistentClient(path="../chroma_db")

In [3]:
import pandas as pd

# 관세법
law_df = pd.read_csv("../output/customs_law_chunks.csv",dtype=str)
law_df["source_type"] = "customs_law"

# 한중 PSR
cn_psr_df = pd.read_csv("../output/korus_cn_psr_chunks.csv",dtype=str)
cn_psr_df["source_type"] = "psr"
cn_psr_df["agreement"] = "KOREA_CHINA_FTA"

# 한미 PSR
us_psr_df = pd.read_csv("../output/korus_psr_chunks.csv",dtype=str)
us_psr_df["source_type"] = "psr"
us_psr_df["agreement"] = "KORUS_FTA"

# 한중 통관
cn_customs_df = pd.read_csv("../output/kr_cn_customs_facilitation.csv",dtype=str)
cn_customs_df["source_type"] = "fta"

# 한미 통관
us_customs_df = pd.read_csv("../output/kr_us_customs_facilitation.csv",dtype=str)
us_customs_df["source_type"] = "fta"

df = pd.concat([
    law_df,
    cn_psr_df,
    us_psr_df,
    cn_customs_df,
    us_customs_df
], ignore_index=True)

print(df.shape)

(6328, 7)


In [4]:
print(df.columns)

Index(['article', 'content', 'source_type', 'code', 'agreement', 'chapter',
       'title'],
      dtype='object')


In [5]:
for col in [
    "article",
    "content",
    "code",
    "agreement",
    "chapter",
    "title",
    "source_type"
]:
    if col not in df.columns:
        df[col] = ""

df = df.fillna("")

In [6]:
# [개선] article/title/code가 비어 있는 행에서 빈 줄이 그대로 들어가던 노이즈 제거
# (원본: "\n" + "\n" + "\n" + content 형태로 빈 줄이 그대로 텍스트에 포함됨)
FIELDS_FOR_EMBEDDING = ["article", "title", "code", "content"]

def build_embedding_text(row):
    parts = [str(row[f]).strip() for f in FIELDS_FOR_EMBEDDING if str(row[f]).strip()]
    return "\n".join(parts)

df["embedding_text"] = df.apply(build_embedding_text, axis=1)

In [7]:
import torch
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

# [가장 중요한 수정] 원본 코드는 collection 생성 시 embedding_function을 지정하지 않았습니다.
# 그러면 ChromaDB 기본값인 all-MiniLM-L6-v2(영어 중심 소형 모델)가 한국어 법령/협정문에도
# 그대로 쓰이게 됩니다. "사전심사" 검색에 화학물질 PSR 항목이 뜬 게 바로 이 증상입니다.
# 한국어를 포함한 다국어를 잘 처리하는 BAAI/bge-m3로 교체합니다. (최초 실행 시 모델
# 다운로드(~2.2GB)가 필요하니 인터넷 연결 상태에서 실행하세요)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

embedding_fn = SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-m3",
    device=device,
    normalize_embeddings=True,  # cosine 유사도를 쓰려면 정규화가 필요합니다
)

# 기존 v2는 차원이 다른 모델(384차원)로 만들어졌기 때문에 같은 이름을 그대로 쓰면
# 차원 불일치 에러가 납니다. 비교해보기 위해 새 이름으로 만들고, 결과가 만족스러우면
# 아래 줄의 주석을 풀어 v2를 지우세요.
# client.delete_collection("customs_knowledge_v2")

collection = client.get_or_create_collection(
    "customs_knowledge_v3",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},
)

/Users/seoyeonkim/customs-rag/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/seoyeonkim/customs-rag/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
from tqdm import tqdm

# [개선] 한 행씩 add()를 호출하던 기존 방식은 6,328번의 개별 DB 호출이 발생해 느립니다.
# 200개씩 묶어서 호출 횟수를 줄입니다.
BATCH_SIZE = 200

ids_all = df.index.astype(str).tolist()
docs_all = df["embedding_text"].tolist()
metas_all = [
    {
        "source_type": row.source_type,
        "agreement": row.agreement,
        "article": row.article,
        "title": row.title,
        # code를 항상 문자열로 고정합니다. 그렇지 않으면 0506.10 같은 코드가
        # 506.1처럼 부동소수점으로 잘못 저장될 수 있습니다 (아래 점검 참고).
        "code": str(row.code) if row.code else "",
    }
    for row in df.itertuples()
]

for i in tqdm(range(0, len(ids_all), BATCH_SIZE)):
    collection.add(
        ids=ids_all[i : i + BATCH_SIZE],
        documents=docs_all[i : i + BATCH_SIZE],
        metadatas=metas_all[i : i + BATCH_SIZE],
    )

print(collection.count())

100%|██████████| 32/32 [01:03<00:00,  1.98s/it]

6328


In [9]:
result = collection.query(
    query_texts=["사전심사"],
    n_results=10,
)

for doc, meta, dist in zip(
    result["documents"][0],
    result["metadatas"][0],
    result["distances"][0],
):
    print(f"distance={dist:.4f}")
    print(meta)
    print(doc[:300])
    print("=" * 80)

distance=0.6034
{'article': '', 'agreement': 'KOREA_CHINA_FTA', 'title': '', 'code': '4202.29', 'source_type': 'psr'}
4202.29
4202.29 
기타 
4 단위 세번변경기준 
 
 
 
통상 포켓이나 핸드백에 넣어 다니는 
물품
distance=0.6109
{'article': '', 'code': '4401.22', 'agreement': 'KOREA_CHINA_FTA', 'source_type': 'psr', 'title': ''}
4401.22
4401.22 
활엽수류 
2 단위 세번변경기준 
 
 
 
톱밥, 목재의 웨이스트(waste)와 
스크랩(scrap)[통나무ㆍ브리케트(briq
uette)ㆍ펠릿(pellet)이나 이와 유사한 
모양으로 응결된 것인지에 상관없다]
distance=0.6119
{'source_type': 'psr', 'code': '6117.90', 'agreement': 'KOREA_CHINA_FTA', 'title': '', 'article': ''}
6117.90
6117.90 
부분품 
2 단위 세번변경기준 
또는 체약 당사국내에서 
발생한 부가가치가 
40 퍼센트 이상일 것 
62 
 
 
제62 류 의류와 그 부속품(메리야스 
편물이나 뜨개질편물은 제외한다) 
 
 
62.01 
 
남성용이나 소년용 오버코트(overcoat)ㆍ 
카코트(car-coat)ㆍ케이프(cape)ㆍ 
클록(cloak)ㆍ아노락(anorak) 
(스키재킷을 포함한다)ㆍ 
윈드치터(wind-cheater)ㆍ 
윈드재킷(wind-jacket)과 이와 유사한 
의류(제6203 호의 것은 제외한다) 
 
 
distance=0.6121
{'article': '', 'code': '8714.20', 'source_type': 'psr', 'agreement': 'KOREA_CHINA_FTA', 'title': ''}
8714.20
8714.20 
신체장애인용 차량

In [15]:
target_idx = df[df["article"].str.contains("제86조", na=False)].index[0]
target_id = str(target_idx)
print("target:", df.loc[target_idx, "article"][:50])

res = collection.query(query_texts=["사전심사"], n_results=collection.count())
ids_ranked = res["ids"][0]

if target_id in ids_ranked:
    rank = ids_ranked.index(target_id) + 1
    dist = res["distances"][0][rank - 1]
    print(f"제86조 순위: {rank} / {len(ids_ranked)}, distance: {dist:.4f}")
else:
    print("결과에 아예 없음")

target: 제86조(특정물품에 적용될 품목분류의 사전심사) ① 다음 각 호의 어느 하나에 해당하는 자
제86조 순위: 1777 / 6328, distance: 0.6497


In [17]:
res = collection.query(
    query_texts=["사전심사"],
    n_results=20
)

for rank, (doc, dist) in enumerate(
    zip(res["documents"][0], res["distances"][0]),
    start=1
):
    print(rank, dist)
    print(doc[:200])
    print("-"*80)

1 0.603384792804718
4202.29
4202.29 
기타 
4 단위 세번변경기준 
 
 
 
통상 포켓이나 핸드백에 넣어 다니는 
물품
--------------------------------------------------------------------------------
2 0.6109117269515991
4401.22
4401.22 
활엽수류 
2 단위 세번변경기준 
 
 
 
톱밥, 목재의 웨이스트(waste)와 
스크랩(scrap)[통나무ㆍ브리케트(briq
uette)ㆍ펠릿(pellet)이나 이와 유사한 
모양으로 응결된 것인지에 상관없다]
--------------------------------------------------------------------------------
3 0.6119086742401123
6117.90
6117.90 
부분품 
2 단위 세번변경기준 
또는 체약 당사국내에서 
발생한 부가가치가 
40 퍼센트 이상일 것 
62 
 
 
제62 류 의류와 그 부속품(메리야스 
편물이나 뜨개질편물은 제외한다) 
 
 
62.01 
 
남성용이나 소년용 오버코트(overcoat)ㆍ 
카코트(car-coat)ㆍ케이프(cape)ㆍ 
클록(cloak)ㆍ아
--------------------------------------------------------------------------------
4 0.6120930910110474
8714.20
8714.20 
신체장애인용 차량의 것 
체약 당사국내에서 
발생한 부가가치가 
40 퍼센트 이상일 것 
 
 
 
기타
--------------------------------------------------------------------------------
5 0.6121747493743896
0802.12
0802.12 
껍데기를 벗긴 것 
완전생산기준 
 
 
 
헤즐너트(hazelnut)나 필버트(filbert) 
[코리루스(Corylus)종] 
 
 


## 다음 단계 (선택)
아래는 임베딩 모델 교체만으로 부족할 때 추가로 시도해볼 만한 것들입니다.

In [10]:
# [선택] 질의에 HS코드나 조문 번호가 그대로 포함되어 있으면, where_document의
# $contains로 그 문자열이 정확히 포함된 문서를 우선적으로 가져올 수 있습니다.
# 벡터 검색만으로는 짧은 코드/번호의 정확한 일치를 놓치는 경우가 있어 보완용으로 유용합니다.
exact_result = collection.query(
    query_texts=["8517.12 원산지 결정기준"],
    n_results=10,
    where_document={"$contains": "8517.12"},
)

In [12]:
# [선택] 벡터 검색 top-k가 아직 부정확하다면 Claude로 후보를 재정렬(rerank)하는
# 단계를 추가하는 것도 효과적입니다. 이미 LangGraph + Claude API를 쓰고 있으니
# 그 파이프라인에 노드 하나로 바로 끼워넣을 수 있습니다.
import anthropic

claude = anthropic.Anthropic()  # ANTHROPIC_API_KEY 환경변수 필요

def rerank_with_claude(query, candidates, top_k=5):
    numbered = "\n\n".join(f"[{i}] {c['text'][:300]}" for i, c in enumerate(candidates))
    prompt = f"""다음은 관세 분류 질의와 벡터 검색으로 찾은 후보 문서입니다.

질의: {query}

후보 문서:
{numbered}

질의와 가장 관련 있는 후보를 관련도 순으로 최대 {top_k}개 골라 번호만
쉼표로 구분해서 출력하세요. 다른 설명은 출력하지 마세요."""

    resp = claude.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=100,
        messages=[{"role": "user", "content": prompt}],
    )
    order = [int(x) for x in resp.content[0].text.strip().split(",") if x.strip().isdigit()]
    return [candidates[i] for i in order if i < len(candidates)]


top20 = collection.query(query_texts=["사전심사"], n_results=20)
candidates = [
    {"id": i, "text": doc, "metadata": meta}
    for i, doc, meta in zip(top20["ids"][0], top20["documents"][0], top20["metadatas"][0])
]
for c in rerank_with_claude("사전심사", candidates, top_k=5):
    print(c["metadata"])
    print(c["text"][:200])
    print("=" * 80)

ModuleNotFoundError: No module named 'anthropic'

In [11]:
# [선택] 임베딩 모델 교체 전/후 비교용 간단 점검 쿼리.
# 정답 문서 라벨이 따로 없어 source_type만 보는 약식 체크지만, 모델을 바꿀 때마다
# 돌려보면 개선 여부를 빠르게 감지할 수 있습니다.
sanity_queries = {
    "사전심사": "customs_law",
    "원산지 결정기준": "psr",
    "관세 가산세": "customs_law",
    "통관 절차": "fta",
}

for q, expected in sanity_queries.items():
    res = collection.query(query_texts=[q], n_results=5)
    hit_types = [m["source_type"] for m in res["metadatas"][0]]
    ok = expected in hit_types
    print(f"{'OK  ' if ok else 'MISS'} {q:12s} top5 source_type={hit_types}")

MISS 사전심사         top5 source_type=['psr', 'psr', 'psr', 'psr', 'psr']
OK   원산지 결정기준     top5 source_type=['psr', 'psr', 'psr', 'psr', 'psr']
MISS 관세 가산세       top5 source_type=['psr', 'psr', 'psr', 'psr', 'psr']
MISS 통관 절차        top5 source_type=['psr', 'psr', 'psr', 'psr', 'psr']
